# 06 - Local RAG over project documents

This example searches both `README.md` and `docs/sensor_caveats.md` with `embeddinggemma`, then asks `qwen3:4b` to answer from the best matching passage.

The demo question checks whether zero sensor readings automatically mean missing data or a fault. Run the code cell from top to bottom.


In [1]:
from pathlib import Path
import numpy as np
import ollama

# Find the project folder, whether this notebook starts in the project root or notebooks/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "README.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find README.md. Open this notebook from the project folder.")

# Read both documents. Keep each bullet as its own passage and attach its section heading.
document_paths = [
    ("README.md", PROJECT_ROOT / "README.md"),
    ("docs/sensor_caveats.md", PROJECT_ROOT / "docs" / "sensor_caveats.md"),
]
passages = []
for source_name, path in document_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Required RAG document not found: {path}")

    text = path.read_text(encoding="utf-8")
    current_heading = ""
    paragraph_lines = []

    def save_paragraph():
        paragraph = " ".join(paragraph_lines).strip()
        if len(paragraph) > 40:
            full_text = f"{current_heading}\n{paragraph}" if current_heading else paragraph
            passages.append({"source": source_name, "text": full_text})
        paragraph_lines.clear()

    for line in text.splitlines():
        line = line.strip()
        if line.startswith("#"):
            save_paragraph()
            current_heading = line.lstrip("# ").strip()
        elif line.startswith("- "):
            save_paragraph()
            bullet = line[2:].strip()
            full_text = f"{current_heading}\n{bullet}" if current_heading else bullet
            if len(full_text) > 40:
                passages.append({"source": source_name, "text": full_text})
        elif not line:
            save_paragraph()
        else:
            paragraph_lines.append(line)
    save_paragraph()

if not passages:
    raise ValueError("No searchable passages were found in the documents")

# Ask about sensor zeros. EmbeddingGemma represents the passages and question as vectors.
question = "Do zero sensor readings always mean that data is missing or the turbine has a fault?"
EMBEDDING_MODEL = "embeddinggemma"
CHAT_MODEL = "qwen3:4b"

def embed_text(text):
    result = ollama.embed(model=EMBEDDING_MODEL, input=text)
    return np.asarray(result.embeddings[0], dtype=np.float32)

document_vectors = np.vstack([embed_text(item["text"]) for item in passages])
question_vector = embed_text(question)

# Compare the vectors and retrieve the closest passage.
denominators = np.linalg.norm(document_vectors, axis=1) * np.linalg.norm(question_vector)
similarities = (document_vectors @ question_vector) / np.maximum(denominators, 1e-12)
best_index = int(np.argmax(similarities))
source = passages[best_index]["source"]
context = passages[best_index]["text"]

print("Retrieved from:", source)
print("Similarity score:", round(float(similarities[best_index]), 3))
print("Retrieved passage:\n", context)

# Ask Qwen to answer simply using the retrieved passage only.
reply = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Answer in one friendly, beginner-friendly sentence using only the provided passage. "
                "If the passage is about zero sensor values, mention the possibilities it lists and say "
                "not to assume zero means a fault or missing data. If the passage does not answer, say that."
            ),
        },
        {"role": "user", "content": f"Passage:\n{context}\n\nQuestion: {question}"},
    ],
)
print("\nAnswer:")
print(reply.message.content)


Retrieved from: docs/sensor_caveats.md
Similarity score: 0.612
Retrieved passage:
 General SCADA readings
Zero values may be real, a stopped-state value, a communication default, or a missing-value encoding. Check operating state and neighboring measurements before treating zero as a confirmed fault.

Answer:
Zero sensor readings don't always mean data is missing or the turbine has a fault—they could be real values, a stopped state, communication defaults, or missing-value encodings, so always check the operating state and neighboring measurements first.
